In [1]:
import torch
import torch.nn as nn
from transformers import RobertaModel, RobertaTokenizer
from transformers import RobertaConfig
import numpy as np
import pandas as pd

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class Roberta_Classifier(nn.Module):
    def __init__(self,
                 freeze_bert=False,
                 bert_dropout=0.3,
                 classifier_dropout=0.3):
        super(Roberta_Classifier, self).__init__()

        n_input = 768
        n_hidden = 50
        n_output = 5

        self.roberta = RobertaModel.from_pretrained("roberta-base")

        self.classifier = nn.Sequential(
            nn.Linear(n_input, n_hidden),
            nn.Dropout(0.2),
            nn.LeakyReLU(),
            nn.Linear(n_hidden, n_output)
        )

        if freeze_bert:
            for param in self.roberta.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        # Feed input data (input_ids and attention_mask) to BERT
        outputs = self.roberta(input_ids=input_ids,
                            attention_mask=attention_mask)

        # Extract the last hidden state of the `[CLS]` token from the BERT output (useful for classification tasks)
        last_hidden_state_cls = outputs[0][:, 0, :]

        # Feed the extracted hidden state to the classifier to compute logits
        logits = self.classifier(last_hidden_state_cls)

        return logits

In [4]:
model = Roberta_Classifier(freeze_bert=False).to(device)
state = torch.load("roberta_cyberbullying_model.pth", map_location=device)
model.load_state_dict(state, strict=False)
model.eval()

Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'lm_head.decoder.weight', 'lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Roberta_Classifier(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): 

In [5]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

In [32]:
import torch
import torch.nn.functional as F

def classify_text(text, model, tokenizer, max_length=256, device=device):
    model.eval()
    model.to(device)

    encoded = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=-1).cpu().tolist()[0]
        pred_idx = int(torch.argmax(logits, dim=-1).cpu().item())
        pred_label = class_labels[pred_idx]  # **mapare index -> label**

    return pred_label, probs


In [33]:
FILE_PATH = '../../../cyberbullying_tweets_transformers.csv'
df = pd.read_csv(FILE_PATH)
df = df.dropna(how='any')

mapping = df[['label', 'type']].drop_duplicates().sort_values('label')
print(mapping)

       label                 type
0          0               gender
7973       1             religion
15971      2  other_cyberbullying
23786      3                  age
31778      4            ethnicity


In [34]:
class_labels = df.sort_values('label')['type'].unique().tolist()

In [39]:
texts = [
    "Nigger",
    "Little girl",
    "Just tell you like girls",
    "Fuck you lesbian",
    "Fucking Muslim",
    "FUCK YOU!!"
]

for text in texts:
    label, probs = classify_text(
        text,
        model=model,
        tokenizer=tokenizer,
        max_length=256,
        device=device
    )
    print(f"\nText: {text!r}")
    print(f"  Predicted label : {label}")
    print("  Probabilities:")
    for cls, p in zip(class_labels, probs):
        print(f"    {cls:20s}: {p:.4f}")


Text: 'Nigger'
  Predicted label : ethnicity
  Probabilities:
    gender              : 0.0001
    religion            : 0.0006
    other_cyberbullying : 0.0003
    age                 : 0.0007
    ethnicity           : 0.9984

Text: 'Little girl'
  Predicted label : age
  Probabilities:
    gender              : 0.0610
    religion            : 0.0031
    other_cyberbullying : 0.0539
    age                 : 0.8794
    ethnicity           : 0.0026

Text: 'Just tell you like girls'
  Predicted label : gender
  Probabilities:
    gender              : 0.9947
    religion            : 0.0004
    other_cyberbullying : 0.0034
    age                 : 0.0013
    ethnicity           : 0.0002

Text: 'Fuck you lesbian'
  Predicted label : gender
  Probabilities:
    gender              : 0.9973
    religion            : 0.0004
    other_cyberbullying : 0.0015
    age                 : 0.0006
    ethnicity           : 0.0002

Text: 'Fucking Muslim'
  Predicted label : religion
  Probabilitie